In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan
spark = SparkSession.builder.appName("SupplyChain").getOrCreate()

raw_path = "/Volumes/workspace/default/supply_chain_data/DataCoSupplyChainDataset.csv"

df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .option("encoding", "ISO-8859-1")
      .csv(raw_path))

print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

Rows: 180519
Columns: 53
root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer City: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer Fname: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Lname: string (nullable = true)
 |-- Customer Password: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Customer State: string (nullable = true)
 |-- Customer Street: string (nullable = true)
 |-- Customer Zipcode: integer (nullable = true)
 |-- Department

In [0]:
# Rename columns to make them Delta-compatible
for col_name in df.columns:
    new_col_name = col_name.replace(" ", "_").replace("(", "").replace(")", "")
    df = df.withColumnRenamed(col_name, new_col_name)

print("Null Counts:")
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df.columns
]).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_%"] = (null_counts["null_count"] / df.count() *100).round(1)
print(null_counts[null_counts["null_count"] > 0].to_string())

dupes = df.count() - df.dropDuplicates().count()
print(f"Duplicate Rows: {dupes}")

print("\n Target: Order_Item_Quantity :")
df.select("Order_Item_Quantity").describe().show()

df.write.format("delta").mode("overwrite") \
  .save("/Volumes/workspace/default/supply_chain_data/delta/raw")

print("Data written to delta")

Null Counts:
                     null_count  null_%
Customer_Lname                8     0.0
Customer_Zipcode              3     0.0
Order_Zipcode            155679    86.2
Product_Description      180519   100.0
Duplicate Rows: 0

 Target: Order_Item_Quantity :
+-------+-------------------+
|summary|Order_Item_Quantity|
+-------+-------------------+
|  count|             180519|
|   mean|  2.127637533999191|
| stddev|   1.45345148142264|
|    min|                  1|
|    max|                  5|
+-------+-------------------+

Data written to delta


dropping order_zipcode and and product_description due to the nulls 